In [2]:
import pandas as pd
import numpy as np
import re


In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
input_file = PROJECT_ROOT / "Data" / "RawTXT" / "2023 Album Reviews.txt"



    

In [5]:


# Read the file
# utf-8-sig automatically removes a UTF-8 BOM if one exists
with open(input_file, "r", encoding="utf-8-sig") as f:
    text = f.read()

header_pattern = re.compile(
    r'(?m)^\s*-?\d+/\d+\s*:?\s+[“"](.+?)[”"]\s+by\s+(.+?)\s*$'
)

headers = list(header_pattern.finditer(text))

print("Headers found:", len(headers))

rows = []

for i, header in enumerate(headers):

    # Get everything from this header to the next header
    start = header.start()
    end = headers[i + 1].start() if i + 1 < len(headers) else len(text)

    review = text[start:end]

    # Correct capture groups:
    # group(1) = album name
    # group(2) = artist name
    album_name = header.group(1).strip()
    artist_name = header.group(2).strip()

    # Everything after the header
    notes_and_score = review[header.end() - start:].strip()

    # Score is the final number/number
    score_match = re.search(
        r'(\d+(?:\.\d+)?/\d+)\s*$',
        notes_and_score
    )

    if score_match:
        score = score_match.group(1)
        notes = notes_and_score[:score_match.start()].strip()
    else:
        score = None
        notes = notes_and_score

    rows.append({
        "album name": album_name,
        "artist name": artist_name,
        "score": score,
        "notes": notes
    })

df = pd.DataFrame(rows)

print(df.shape)
print(df.head())




Headers found: 259
(259, 4)
          album name     artist name   score  \
0              Lahai          Sampha  8.5/10   
1  melodic nostalgia       tomcbumpz    5/10   
2       OK ORCHESTRA             AJR    1/10   
3           HELLMODE  Jeff Rosentock    8/10   
4     Dorothy's Harp   Dorothy Ashby  9.5/10   

                                               notes  
0  A smooth R&B infused hip hop album. Opener is ...  
1  This is a lowfi beats to study to ep. I didn't...  
2  I decided I need to suffer I guess. AJR is the...  
3  The dying masterpiece of California garage pun...  
4  A funky bass and drum beat lay the ground for ...  


In [105]:
print(text.splitlines()[0])
print([ord(c) for c in text.splitlines()[0]])

260/260 “Lahai” by Sampha
[50, 54, 48, 47, 50, 54, 48, 32, 8220, 76, 97, 104, 97, 105, 8221, 32, 98, 121, 32, 83, 97, 109, 112, 104, 97]


In [106]:
headers = list(header_pattern.finditer(text))
# Extract the album numbers that were successfully found
found_numbers = []

for header in headers:
    # Get the text immediately before the album title
    line = header.group(0)
    number_match = re.match(r'\s*(\d+)/(\d+)', line)

    if number_match:
        found_numbers.append(int(number_match.group(1)))

# Find which album numbers are missing
expected_numbers = set(range(1, 261))
missing_numbers = sorted(expected_numbers - set(found_numbers))

print("Headers found:", len(headers))
print("Missing album numbers:", missing_numbers)


Headers found: 259
Missing album numbers: [237]


In [107]:
print(df.shape)

print(df.tail(3))

(259, 4)
               album name          artist name   score  \
256      View With A Room         Julian Lage:    8/10   
257     Tell Me In Pretty   Cage the Elephant:  5.5/10   
258  Des Todas Las Flores  Natalia Lafourcade:    9/10   

                                                 notes  
256  Guitar jazz album. Lots of technical skill. Fe...  
257  A 2010's rock album. Reminds me of worse "Whit...  
258  A beautiful Mexican jazz/folk album. Just cons...  


In [108]:
review_numbers = []

for header in headers:
    match = re.match(r'\s*-?(\d+)/(\d+)', header.group(0))
    if match:
        review_numbers.append(int(match.group(1)))

print("Reviews parsed:", len(review_numbers))

# Show the range of review numbers
print("Highest review number:", max(review_numbers))
print("Lowest review number:", min(review_numbers))

# Find gaps
expected = set(range(min(review_numbers), max(review_numbers) + 1))
found = set(review_numbers)

missing = sorted(expected - found)

print("Missing review numbers:", missing)


Reviews parsed: 259
Highest review number: 260
Lowest review number: 1
Missing review numbers: [237]


In [109]:
df.columns

Index(['album name', 'artist name', 'score', 'notes'], dtype='str')

In [110]:
df.rename(columns={"album name": "album_name", "artist name": "artist_name"}, inplace=True)
#------
# Make sure everything is treated as a string
df['score'] = df['score'].astype(str)

# Convert scores out of 3
df.loc[df['score'].str.endswith('/3'), 'score'] = (
    df.loc[df['score'].str.endswith('/3'), 'score']
    .str.replace('/3', '', regex=False)
    .astype(float) * 2
).astype(str)

# Remove /10
df['score'] = df['score'].str.replace('/10', '', regex=False)

# Convert the entire column to numbers
df['score'] = pd.to_numeric(df['score'])
#-----------------
df.to_csv(
    r"C:\Users\alexw\Projects\ALBUM_PROJ\2023_reviews.csv",
    index=False,
    encoding="utf-8"
)


In [111]:
df.head(100)

,album_name,artist_name,score,notes
0,Lahai,Sampha,8.5,A smooth R&B infused hip hop album. Opener is ...
1,melodic nostalgia,tomcbumpz,5.0,This is a lowfi beats to study to ep. I didn't...
2,OK ORCHESTRA,AJR,1.0,I decided I need to suffer I guess. AJR is the...
3,HELLMODE,Jeff Rosentock,8.0,The dying masterpiece of California garage pun...
4,Dorothy's Harp,Dorothy Ashby,9.5,A funky bass and drum beat lay the ground for ...
...,...,...,...,...
95,North Hills,Dawes,6.5,It's a poppy country ballad affair. First song...
96,Purple,Stone Temple Pilots,6.0,"It's a 1994 punk album. It's cool, but it waiv..."
97,They Live In My Head,Bush Tetras,5.5,Starts on a very pixieesque note. There's the ...
98,Love Yourself,BTS,3.0,Yep. Starts really bland modern edm popnb. It'...


In [112]:
with open(
    r"C:\Users\alexw\Projects\ALBUM_PROJ\2023_reviews.csv",
    encoding="utf-8"
) as f:
    f.read()

print("UTF-8 is valid")

UTF-8 is valid
